# Web Scraping with Python — Part 1
### From a Website to Your Own CSV File

**Instructor: Ananda Rimal**

---

## What you will build today

By the end of this notebook, you will write a program that:

1. Visits a real website
2. Reads its HTML
3. Extracts the data you want
4. Saves it as a **CSV file** you can open in Excel

**No prior scraping knowledge needed.** You only need basic Python: variables, lists, loops, and functions.

---

## Our practice website

We will use **https://books.toscrape.com** — a website built *specifically for learning web scraping*. It is 100% legal and safe to scrape.

Open it in a new browser tab now and keep it open. You will need it.

## Step 0 — Setup

Run the cell below **once** to install the libraries we need.

| Library | What it does |
|---|---|
| `requests` | Downloads webpages |
| `beautifulsoup4` | Searches inside HTML |
| `pandas` | Organizes data into tables |
| `openpyxl` | Lets pandas save Excel files |

In [1]:
# Run this cell once (in Colab or Jupyter)
%pip install requests beautifulsoup4 pandas openpyxl --quiet

print("Setup complete! You are ready to scrape.")

Setup complete! You are ready to scrape.


---
# Part 1 — What is Web Scraping?

**Web scraping = collecting information from websites with code instead of copy-paste.**

Imagine copying 1,000 product names and prices from a website by hand. Hours of boring work.
A web scraper does it in seconds.

```
Website  →  HTML  →  Python  →  Extract Data  →  CSV / Excel
```

That is the entire journey. Every scraper you will ever write follows these 5 steps.

---
# Part 2 — Look Inside a Website (Inspect Element)

Before writing ANY code, we always look at the website's HTML first.

### 🔍 Activity — do this now (no code needed):

1. Open **https://books.toscrape.com** in your browser
2. Move your mouse over any **book price** (like £51.77)
3. **Right-click** → choose **Inspect**
4. Look at the highlighted line in the panel that opens

You should see something like this:

```html
<p class="price_color">£51.77</p>
```

### The 3 things you need to understand:

| Concept | Example | Meaning |
|---|---|---|
| **Tag** | `<p>`, `<h3>`, `<a>` | The type of element |
| **Class** | `class="price_color"` | A label attached to the tag |
| **Content** | `£51.77` | The text inside |

> 💡 **The class is our treasure map.** When we scrape, we tell Python:
> *"Find the element with class `price_color` and give me its text."*

**✅ Checkpoint:** Before moving on, use Inspect to find the class name of a book **title** container. (Hint: hover over the whole book card, not just the title text.)

---
# Part 3 — Your First Request

Time to make Python visit the website for us. Just 3 lines.

In [2]:
import requests

url = "https://books.toscrape.com"
response = requests.get(url)

print(response.status_code)

200


### What does `200` mean?

The **status code** is the server's answer to our request:

| Code | Meaning |
|---|---|
| **200** | ✅ Success — we got the page |
| 404 | ❌ Page not found |
| 403 | ❌ Access forbidden |
| 500 | ❌ Server error |

**Rule: always check for 200 before scraping.**

Let's break it on purpose to see a 404:

In [3]:
bad_response = requests.get("https://books.toscrape.com/this-page-does-not-exist")
print(bad_response.status_code)   # 404 — page not found

404


### Now let's see what we actually downloaded

In [ ]:
# The HTML of the page is stored in response.text
# Let's peek at the first 500 characters:

print(response.text[:500])

😵 **That's a mess, right?**

The full page is **thousands** of lines like this. All our data (titles, prices) is buried in there — but nobody wants to search it by hand.

**That is exactly why BeautifulSoup exists.**

---
# Part 4 — BeautifulSoup: Making HTML Searchable

BeautifulSoup takes the messy HTML text and turns it into an object we can **search**.

In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

print(type(soup))   # it's a BeautifulSoup object now, not plain text

<class 'bs4.BeautifulSoup'>


### `find()` — get the FIRST matching element

Let's grab the main heading of the page:

In [5]:
heading = soup.find("h1")

print(heading)        # the whole element, with tags
print(heading.text)   # only the text inside

<h1>All products</h1>
All products


> 💡 **`.text` is your best friend.** The element comes wrapped in HTML tags —
> `.text` strips the tags and gives you just the content.

### `find_all()` — get ALL matching elements (as a list)

In [6]:
all_h3 = soup.find_all("h3")

print("How many <h3> elements?", len(all_h3))
print()
print(all_h3[0])   # the first one — each book title lives inside an <h3>

How many <h3> elements? 20

<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>


### Searching by class — using the treasure map from Part 2

Remember the price element you inspected? `<p class="price_color">`

⚠️ **Important:** in Python we write `class_` with an **underscore**
(because `class` alone is a reserved word in Python).

In [7]:
first_price = soup.find("p", class_="price_color")

print(first_price.text)

Â£51.77


### ⚠️ The most common beginner error — let's see it now

What happens if we search for something that doesn't exist?

In [8]:
result = soup.find("p", class_="this_class_does_not_exist")

print(result)   # None — nothing found!

# Uncommenting the next line would crash with:
# AttributeError: 'NoneType' object has no attribute 'text'
# print(result.text)

None


> 🧠 **Remember this error.** `'NoneType' object has no attribute 'text'` means:
> **"your search found nothing"** — usually a typo in the tag or class name.
> When you see it, go back to Inspect Element and check the exact spelling.

---
# Part 5 — Scraping One Complete Book

Each book on the page lives inside:

```html
<article class="product_pod">
    <h3><a title="A Light in the Attic" href="...">A Light in ...</a></h3>
    <p class="price_color">£51.77</p>
    ...
</article>
```

Let's grab **one** book first and extract everything from it:

In [9]:
# Step 1: grab the first book card
book = soup.find("article", class_="product_pod")

# Step 2: the full title is stored in the <a> tag's "title" attribute
title = book.find("h3").find("a")["title"]

# Step 3: the price is the text of the price element
price = book.find("p", class_="price_color").text

print("Title:", title)
print("Price:", price)

Title: A Light in the Attic
Price: Â£51.77


> 💡 **New trick:** `element["title"]` reads an **attribute** of a tag,
> while `.text` reads the visible content. Here the full book title is
> stored as an attribute: `<a title="A Light in the Attic">`.

**If this works for ONE book, a loop makes it work for ALL books.** That's next.

---
# Part 6 — The Full Scraper: Loop → Table → CSV 🎉

This is the moment everything comes together.

In [10]:
import pandas as pd

# 1. Find ALL book cards on the page
books = soup.find_all("article", class_="product_pod")
print("Books found:", len(books))

# 2. Loop through them and collect the data
data = []

for book in books:
    title = book.find("h3").find("a")["title"]
    price = book.find("p", class_="price_color").text
    availability = book.find("p", class_="instock availability").text.strip()

    data.append({
        "title": title,
        "price": price,
        "availability": availability
    })

# 3. Turn the list into a table
df = pd.DataFrame(data)
df.head(10)   # show the first 10 rows

Books found: 20


,title,price,availability
0,A Light in the Attic,Â£51.77,In stock
1,Tipping the Velvet,Â£53.74,In stock
2,Soumission,Â£50.10,In stock
3,Sharp Objects,Â£47.82,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock
5,The Requiem Red,Â£22.65,In stock
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,In stock
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,In stock
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,In stock
9,The Black Maria,Â£52.15,In stock


### Save it as a file you can open in Excel:

In [11]:
df.to_csv("books.csv", index=False)
df.to_excel("books.xlsx", index=False)

print("Saved! Look for books.csv and books.xlsx in your file panel.")
# In Colab: click the folder icon on the left to download them.

Saved! Look for books.csv and books.xlsx in your file panel.


## 🏆 Congratulations!

You just built a complete web scraper:

```
Website → requests.get() → BeautifulSoup → find_all() → loop → DataFrame → CSV
```

Read that pipeline again. **Every scraper you ever write will follow it.**